In [ ]:
%%writefile vnc_final.go
package main

import (
	"fmt"
	"math/rand"
	"net"
	"net/http"
	"net/url"
	"strings"
	"time"
)

const (
	BOT_TOKEN = "8387293970:AAF8p2KglOGvKvzHtruOnKdFxkAGCpVGqFo"
	CHAT_ID   = "8357044091"
	TIMEOUT   = 3 * time.Second
)

var (
	WORKERS = 1500
)

func main() {
	rand.Seed(time.Now().UnixNano())
	fmt.Printf("\x1b[33m[!] GFI VNC ALL-IN-ONE | HUNTING NO-PASS & EASY-PASS | ACTIVE\x1b[0m\n")

	jobs := make(chan string, WORKERS*2)
	for i := 0; i < WORKERS; i++ {
		go worker(jobs)
	}

	for {
		// Quét ngẫu nhiên toàn cầu
		ip := fmt.Sprintf("%d.%d.%d.%d", rand.Intn(223)+1, rand.Intn(255), rand.Intn(255), rand.Intn(255))
		jobs <- net.JoinHostPort(ip, "5900")
		time.Sleep(400 * time.Microsecond)
	}
}

func worker(jobs chan string) {
	for target := range jobs {
		checkVNCDetails(target)
	}
}

func checkVNCDetails(target string) {
	conn, err := net.DialTimeout("tcp", target, TIMEOUT)
	if err != nil { return }
	defer conn.Close()

	// 1. Đọc Banner RFB
	buf := make([]byte, 12)
	conn.SetReadDeadline(time.Now().Add(TIMEOUT))
	n, err := conn.Read(buf)
	if err != nil || n < 3 || !strings.HasPrefix(string(buf), "RFB") {
		return
	}

	// 2. Phản hồi banner để lấy Security Types
	conn.Write([]byte("RFB 003.003\n"))

	// 3. Đọc Security Types (Số lượng và danh sách)
	secBuf := make([]byte, 32)
	n, err = conn.Read(secBuf)
	if err != nil || n == 0 { return }

	isNoAuth := false
	isVNCAuth := false

	// Duyệt qua các Security Types mà server hỗ trợ
	// Type 1: None (Không mật khẩu)
	// Type 2: VNC Authentication (Có mật khẩu)
	for i := 0; i < n; i++ {
		if secBuf[i] == 1 { isNoAuth = true }
		if secBuf[i] == 2 { isVNCAuth = true }
	}

	if isNoAuth {
		msg := fmt.Sprintf("🎁 [VNC NO-PASS] %s | VAO THANG KHONG CAN MK!", target)
		sendToTele(msg, "31") // Màu đỏ cho hàng cực phẩm
	} else if isVNCAuth {
		msg := fmt.Sprintf("🔑 [VNC AUTH] %s | CO MAT KHAU (THU 123456/1234567)", target)
		sendToTele(msg, "32") // Màu xanh cho hàng có pass
	}
}

func sendToTele(msg string, color string) {
	fmt.Printf("\x1b[%sm\n%s\x1b[0m\n", color, msg)
	apiURL := fmt.Sprintf("https://api.telegram.org/bot%s/sendMessage?chat_id=%s&text=%s", 
		BOT_TOKEN, CHAT_ID, url.QueryEscape(msg))
	go http.Get(apiURL)
}

In [ ]:
# 1. Dọn dẹp repo lỗi
!sudo rm -f /etc/apt/sources.list.d/cran*.list
!sudo sed -i '/cloud.r-project.org/d' /etc/apt/sources.list

# 2. Cập nhật và cài đặt Golang
!sudo apt-get update --fix-missing
!sudo apt-get install golang -y

# 3. Khởi tạo Module và nạp thư viện
!rm -f go.mod go.sum
!go mod init gfi
!go get golang.org/x/crypto@v0.17.0
!go get golang.org/x/sys@v0.15.0

!go run vnc_scan.go

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease                     
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease               
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 99.3 kB in 1s (91.9 kB/s)
Reading pac